# Aspect-Based Sentiment Analysis for Restaurant Ranking

## Load Data

In [55]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

# Read raw data
master_dir = os.path.dirname(os.getcwd())

file_google = os.path.join(master_dir, "data", "raw_data", "GoogleReview_data.csv")
file_tripadvisor = os.path.join(master_dir, "data", "raw_data", "TripAdvisor_data.csv")

df_google = pd.read_csv(file_google)
df_tripadvisor = pd.read_csv(file_tripadvisor)

## Preprocessing

In [56]:
df_google.count()

Author        222020
Rating        222020
Review        222020
Restaurant    222020
Location      222020
dtype: int64

In [57]:
df_tripadvisor.count()

Author        139764
Title         139764
Review        139764
Rating        139764
Dates         139764
Restaurant    139764
Location      139764
dtype: int64

In [58]:
df_google["Platform"] = "Google"
df_tripadvisor["Platform"] = "Tripadvisor"

df = pd.concat(
    [df_google, df_tripadvisor],
    ignore_index=True
)

df.count()

Author        361784
Rating        361784
Review        361784
Restaurant    361784
Location      361784
Platform      361784
Title         139764
Dates         139764
dtype: int64

### Basic Cleaning

In [59]:
import emoji
import re
import contractions

def basic_cleaning(df, keep_emoji_text=True):

    # Convert text to lowercase
    df["Review"] = df["Review"].str.lower()

    # Remove duplicates
    df = df.drop_duplicates()

    #remomve rows with empty or NaN reviews
    df = df[
        df["Review"].notna() &
        df["Review"].astype(str).str.strip().ne("")
    ]
    
    # Remove HTML tags
    df["Review"] = df["Review"].apply(lambda x: re.sub(r'<[^>]+>', '', x))

    # Remove URLs
    df["Review"] = df["Review"].apply(lambda x: re.sub(r'http\S+|www\S+', '[URL]', x))

    # Remove email addresses
    df["Review"] = df["Review"].apply(lambda x: re.sub(r'\S+@\S+', '[EMAIL]', x))

    # Keep or remove emojis based on the parameter 
    if keep_emoji_text:
        df["Review"] = df["Review"].apply(lambda x: emoji.demojize(x, delimiters=(' ', ' ')))
    else:
        df["Review"] = df["Review"].apply(lambda x: emoji.replace_emoji(x, replace=''))

    # Expand contractions
    df["Review"] = df["Review"].apply(lambda x: contractions.fix(x))

    # Remove mentions
    df["Review"] = df["Review"].apply(lambda x: re.sub(r'@\w+', '', x))

    # Normalize whitespace
    df["Review"] = df["Review"].apply(lambda x: re.sub(r'\s+', ' ', x).strip())

    return df

In [60]:
# clean the dataframes
df_cleaned = basic_cleaning(df)

In [61]:
# number of records
sampleNum = df_cleaned.index.size
print(f"Sample Number: {sampleNum}")

Sample Number: 361778


In [62]:
# subset wanted columns
def subset_columns(df):
    df = df[["Review", "Rating", "Platform"]]
    return df  

In [63]:
# remain only the wanted columns
df_cleaned = subset_columns(df_cleaned)

In [64]:
def count_caps_words(text):
    words = re.findall(r'\b[A-Za-z]+\b', text)
    
    return sum(
        1 for word in words
        if len(word) > 1 and word.isupper()
    )

In [65]:
df_cleaned["caps_count"] = df["Review"].apply(count_caps_words)

df_cleaned.head(10)

,Review,Rating,Platform,caps_count
0,came here for the high tea. great service espe...,4.0,Google,0
1,"5 stars for the service, even though some of t...",2.0,Google,0
2,"hi, thank you for your service. but! i feel so...",1.0,Google,0
3,i have the worse buffer dinner ever so far. th...,1.0,Google,0
4,"that is are known 5 elmark "" 9h72 "" & kdk "" 3 ...",5.0,Google,0
5,i just came back from there. 2 adults and 4 yo...,2.0,Google,0
6,restaurant looks nice but taste is bad. i had ...,2.0,Google,0
7,"pros: ambience is great with lake view, good a...",4.0,Google,0
8,we went to this place after reviews on tripadv...,1.0,Google,0
9,"the restaurant is located inside the hotel, th...",4.0,Google,0


### Tokenization

In [66]:
from nltk.tokenize import sent_tokenize, word_tokenize

def tokenize_review(text):
    # Sentence tokenization
    sentences = sent_tokenize(text)
    
    # Word tokenization
    tokens = []
    for sentence in sentences:
        tokens.extend(word_tokenize(sentence))
    
    return sentences, tokens

In [67]:
df_cleaned[["Sentences", "Tokens"]] = df["Review"].apply(
    lambda x: pd.Series(tokenize_review(x))
)

### Stop-word removal

In [68]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords

# Base English stopword list
stop_words = set(stopwords.words("english"))

# Words that carry sentiment, negation, or contrast
important_words = {
    "not",
    "no",
    "nor",
    "never",
    "neither",
    "none",
    "but",
    "however",
    "although",
    "yet",
    "though",
    "very",
    "too",
    "so",
    "really",
    "extremely",
    "quite",
    "highly",
    "somewhat",
    "slightly",
    "badly",
    "hardly",
    "absolutely",
    "completely",
    "could",
    "would",
    "should",
    "might",
    "must"
}

# Keep important words by removing them from stopword list
stop_words = stop_words - important_words

def remove_stopwords(tokens):
    return [
        token for token in tokens
        if token.lower() not in stop_words
    ]

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [69]:
df_cleaned["Tokens_no_stopwords"] = df_cleaned["Tokens"].apply(remove_stopwords)

### Lemmatization

In [70]:
from nltk import pos_tag
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [71]:
def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [72]:
lemmatizer = WordNetLemmatizer()

In [73]:
def lemmatize_tokens(tokens):
    tagged_tokens = pos_tag(tokens)

    lemmatized = []

    for word, tag in tagged_tokens:
        wordnet_pos = get_wordnet_pos(tag)
        lemma = lemmatizer.lemmatize(word, pos=wordnet_pos)
        lemmatized.append(lemma)

    return lemmatized

In [74]:
df_cleaned["Lemmatized_Tokens"] = df_cleaned["Tokens_no_stopwords"].apply(
    lemmatize_tokens
)

In [75]:
df_cleaned["Review_lemmatized"] = df_cleaned["Lemmatized_Tokens"].apply(
    lambda tokens: " ".join(tokens)
)

In [76]:
df_cleaned[[
    "Review",
    "Tokens",
    "Tokens_no_stopwords",
    "Lemmatized_Tokens"
]].head()

,Review,Tokens,Tokens_no_stopwords,Lemmatized_Tokens
0,came here for the high tea. great service espe...,"[came, here, for, the, high, tea, ., great, se...","[came, high, tea, ., great, service, especiall...","[come, high, tea, ., great, service, especiall..."
1,"5 stars for the service, even though some of t...","[5, stars, for, the, service, ,, even, though,...","[5, stars, service, ,, even, though, staffs, n...","[5, star, service, ,, even, though, staff, nee..."
2,"hi, thank you for your service. but! i feel so...","[hi, ,, thank, you, for, your, service, ., but...","[hi, ,, thank, service, ., but, !, feel, so, s...","[hi, ,, thank, service, ., but, !, feel, so, s..."
3,i have the worse buffer dinner ever so far. th...,"[i, have, the, worse, buffer, dinner, ever, so...","[worse, buffer, dinner, ever, so, far, ., spre...","[bad, buffer, dinner, ever, so, far, ., spread..."
4,"that is are known 5 elmark "" 9h72 "" & kdk "" 3 ...","[that, 's, are, known, 5, elmark, ``, 9h72, ``...","['s, known, 5, elmark, ``, 9h72, ``, &, kdk, `...","['s, know, 5, elmark, ``, 9h72, ``, &, kdk, ``..."


In [77]:
# keep the wanted columns only
df_cleaned = df_cleaned[[
    "Review",
    "Rating",
    "caps_count",
    "Lemmatized_Tokens",
    "Review_lemmatized",
    "Platform"
]]

In [78]:
from pathlib import Path

# Find project root
project_dir = Path.cwd()

# Change this if your notebook is inside a subfolder
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

# Create cleaned data directory
cleaned_dir = project_dir / "data" / "cleaned_data"
cleaned_dir.mkdir(parents=True, exist_ok=True)

# Create raw data directory
raw_dir = project_dir / "data" / "raw_data"
raw_dir.mkdir(parents=True, exist_ok=True)

# Export raw file
output_file = raw_dir / "combined_reviews.csv"
df.to_csv(output_file, index=False)

# Export cleaned file
output_file = cleaned_dir / "cleaned_reviews.csv"
df_cleaned.to_csv(output_file, index=False)

print(f"Saved to: {output_file}")

Saved to: c:\Users\user\Downloads\social_computing_assignment\data\cleaned_data\cleaned_reviews.csv
